# Draw — Treinamento SegFormer-B0 (4 classes: background / paper / hand / ink)

**Versão OFFLINE do notebook.** Esta versão NÃO precisa de internet para
rodar as células de preparação: não há `pip install` e não há `git clone`.
Se o kernel do Kaggle rodar sem internet (foi o que derrubou o teste
anterior, com falha de DNS no PyPI e no GitHub), este notebook continua
funcionando — desde que os inputs abaixo estejam anexados.

**O que este notebook faz:** treinar o modelo de segmentação do Draw,
exportar para ONNX e gerar os artefatos de entrega.

**O que este notebook NÃO faz (fora do escopo, Codex depois):**
webcam, integração no app, tracking temporal, memória de traços,
oclusão em tempo real, interface, CAD, simulador.

## Inputs obrigatórios do kernel (Add Input, antes de rodar)

1. **Dataset sintético** — pastas `synthetic/{train,val,test}/{images,masks}`
   (ou o layout plano do gerador) + `split_manifest.json`. É ele que o
   treino usa; já vem com splits por fonte original (mão, folha, fundo)
   — nunca reembaralhe fontes entre train/val/test aqui.
2. **Dataset de código** — uma pasta contendo `training/train.py` e
   `training/config.json` (o conteúdo de `training/` do repo). O notebook
   procura `training/` nos inputs do kernel, nesta ordem:
   `/kaggle/input/*/training`, `/kaggle/input/*/draw/training`,
   `/kaggle/working/draw/training`. **Não há clone do GitHub.**
3. **FreiHAND** (`danieldelro/freihand`) — só para a checagem de formato
   da célula 3. O treino usa o dataset sintético, que já incorpora as
   mãos; a checagem existe para não inventar rótulos de mão.

## GPU e internet

- **GPU:** recomendada para o treino completo (T4 ou P100). Um teste
  pequeno (poucas cenas, `EPOCHS = 2` na célula de treino) roda em CPU,
  mais devagar — serve para validar o notebook, não para treinar o modelo.
- **Internet:** NÃO é necessária para dependências nem para o código.
- **Ressalva honesta — pesos pré-treinados:** o `train.py` carrega
  `nvidia/mit-b0` via `transformers.from_pretrained`, que baixa os pesos
  do HuggingFace na primeira vez se eles não estiverem em cache. Para
  rodar 100% offline, anexe o cache dos pesos como input do kernel ou
  faça uma primeira corrida com internet para cacheá-los. A célula 1
  avisa se o cache não for encontrado, mas não trava o notebook por isso
  (o treino é quem falharia na carga do modelo, com erro do transformers).

## Passo a passo

1. Anexe os 3 inputs acima ao kernel.
2. Rode as células em ordem. As células de localização e de dataset
   **travam com erro claro** se o código ou o dataset não forem achados
   — não seguem com contagens 0 para falhar lá na frente.
3. Para testar, troque só a variável `EPOCHS` no topo da célula de
   treino (ex.: `EPOCHS = 2`); `EPOCHS = None` usa o padrão da config
   (30 épocas). Não é preciso editar o comando de treino.
4. O treino salva tudo em `/kaggle/working/out`.
5. **Tempo estimado:** 30 épocas, ~5-10 mil imagens 512x512, batch 8:
   3-6 h em T4. Dataset menor (2 mil) ou 15 épocas: 1-2 h.


## 1. Verificar o ambiente (sem instalar nada)

O Kaggle já traz torch, transformers, numpy e opencv. Este notebook
**não executa `pip install`**: se uma dependência obrigatória faltar,
a célula abaixo levanta um erro claro dizendo qual falta, em vez de
tentar baixar algo sem internet. `albumentations` é opcional
(o `train.py` treina sem augmentation se ela faltar) e `onnxruntime`
só afeta a validação ONNX (o `train.py` pula essa validação se faltar).


In [ ]:
# Verificação do ambiente — SEM instalar pacotes, SEM download.
# Se algo obrigatório faltar, o erro abaixo diz o que falta; não há
# tentativa de baixar pacotes (o kernel pode estar sem internet).
import importlib, os, sys
from pathlib import Path

REQUIRED = ["torch", "transformers", "cv2", "numpy"]
OPTIONAL = {
    "albumentations": "treino sem augmentation",
    "onnx": "exportação ONNX pode falhar sem ele",
    "onnxruntime": "validação ONNX será pulada pelo train.py",
}
missing = []
for name in REQUIRED:
    try:
        mod = importlib.import_module(name)
        print(f"{name}: OK ({getattr(mod, '__version__', 'sem versão')})")
    except ImportError:
        missing.append(name)
        print(f"{name}: AUSENTE (obrigatório)")
if missing:
    raise RuntimeError(
        "Dependência(s) ausente(s) no ambiente do kernel: " + ", ".join(missing) + ". "
        "Este notebook NÃO instala pacotes (modo offline). Use um ambiente Kaggle "
        "que já traga essa(s) dependência(s), ou anexe-a(s) como input do kernel."
    )
for name, effect in OPTIONAL.items():
    try:
        importlib.import_module(name)
        print(f"{name}: OK (opcional)")
    except ImportError:
        print(f"{name}: ausente (opcional) -> {effect}")

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU: treino completo será muito mais lento; teste pequeno (EPOCHS=2) roda em CPU.")

# Cache dos pesos pré-treinados nvidia/mit-b0 (ressalva de offline).
hf_roots = [Path(os.environ.get("HF_HOME", "")), Path.home() / ".cache" / "huggingface"]
hf_roots += list(Path("/kaggle/input").glob("*/models--nvidia--mit-b0"))
found_cache = any((r / "hub" / "models--nvidia--mit-b0").exists() or r.name == "models--nvidia--mit-b0" for r in hf_roots if str(r))
if found_cache:
    print("cache de pesos nvidia/mit-b0: encontrado")
else:
    print("AVISO: cache de pesos nvidia/mit-b0 NÃO encontrado. O primeiro from_pretrained "
          "tentará baixar do HuggingFace e falhará sem internet. Para 100% offline, "
          "anexe o cache dos pesos como input do kernel ou faça uma corrida com internet para cacheá-los.")


## 2. Localizar código e dataset (sem clone)

O código `training/` vem como **input do kernel**, não do GitHub.
A célula abaixo procura `training/` nos inputs, nesta ordem:
`/kaggle/input/*/training`, `/kaggle/input/*/draw/training`,
`/kaggle/working/draw/training`. Se não achar, ela **levanta erro e para**
— não segue com um caminho falso.


In [ ]:
# Localização do código training/ — SEM clonagem do repo (modo offline).
from pathlib import Path

code_candidates = []
code_candidates += sorted(Path("/kaggle/input").glob("*/training"))
code_candidates += sorted(Path("/kaggle/input").glob("*/draw/training"))
code_candidates.append(Path("/kaggle/working/draw/training"))

REPO = None
for cand in code_candidates:
    if (cand / "train.py").exists():
        REPO = cand.parent
        break
if REPO is None:
    raise RuntimeError(
        "Código training/ NÃO encontrado em nenhum input do kernel. "
        "Anexe o dataset de código com training/train.py + training/config.json "
        "como input do kernel (Add Input). Este notebook NÃO clona o repositório. "
        "Caminhos procurados: /kaggle/input/*/training, /kaggle/input/*/draw/training, "
        "/kaggle/working/draw/training."
    )
print("repo (código):", REPO, "| train.py:", (REPO / "training" / "train.py").exists())


In [ ]:
# Localização do dataset sintético — TRAVA se não achar ou se algum split estiver vazio.
from pathlib import Path

IMAGE_EXTS = {".png", ".jpg", ".jpeg"}
data_candidates = [p for p in Path("/kaggle/input").glob("*/dataset/synthetic")]
data_candidates += [p for p in Path("/kaggle/input").glob("*/synthetic")]
DATA = None
for cand in data_candidates:
    if (cand / "train").is_dir():
        DATA = cand
        break
if DATA is None:
    raise RuntimeError(
        "Dataset sintético NÃO encontrado em nenhum input do kernel. "
        "Anexe o dataset sintético como input do kernel (Add Input). "
        "Caminhos procurados: /kaggle/input/*/dataset/synthetic e /kaggle/input/*/synthetic, "
        "com pelo menos a pasta train/."
    )
print("dataset sintético:", DATA)
counts = {}
for split in ("train", "val", "test"):
    d = DATA / split
    if (d / "images").is_dir():
        n = len([f for f in (d / "images").iterdir() if f.suffix.lower() in IMAGE_EXTS])
    elif d.is_dir():
        # layout plano do gerador: scene_*.png, sem _mask/_ink_full
        n = len([f for f in d.glob("*.png") if not f.stem.endswith(("_mask", "_ink_full"))])
    else:
        n = 0
    counts[split] = n
    print(f"  {split}: {n} imagens")
missing_splits = [s for s, n in counts.items() if n == 0]
if missing_splits:
    raise RuntimeError(
        "Dataset sintético incompleto: split(s) com 0 imagens: " + ", ".join(missing_splits) +
        f" em {DATA}. Corrija o dataset (ou o input do kernel) antes de treinar; "
        "o notebook para aqui em vez de falhar lá na frente, na célula de config."
    )


## 3. Checagem do FreiHAND (de verdade, não só listagem)

Antes do treino, confirme o que a versão do Kaggle realmente contém:
máscara real de mão em `training/mask/`, com dimensão, limiar 127 e
cobertura. **Se `training/mask/` faltar, esta célula levanta erro e o
notebook para aqui** — não gere rótulos de mão sintéticos fingindo que
são reais, e não siga para o treino sem documentar o problema.
O treino em si usa o dataset sintético; esta célula é a verificação
exigida pelo escopo.


In [ ]:
# Checagem real do FreiHAND — PARA com erro claro se training/mask/ faltar.
import cv2
import numpy as np
from pathlib import Path

mask_dirs = sorted(Path("/kaggle/input").glob("*/training/mask"))
mask_dirs += sorted(Path("/kaggle/input").glob("*/*/training/mask"))
mask_dirs = [d for d in dict.fromkeys(mask_dirs) if d.is_dir()]
if not mask_dirs:
    raise RuntimeError(
        "FreiHAND: training/mask/ NÃO encontrado em nenhum input do kernel. "
        "Anexe o dataset danieldelro/freihand como input do kernel (Add Input). "
        "Sem máscara real de mão, NÃO prossiga: documente o problema e troque a fonte — "
        "nunca invente rótulos de mão. Caminhos procurados: /kaggle/input/*/training/mask."
    )
mask_dir = mask_dirs[0]
print("FreiHAND training/mask/:", mask_dir)
mask_files = sorted(mask_dir.glob("*.jpg")) + sorted(mask_dir.glob("*.png"))
if not mask_files:
    raise RuntimeError(f"FreiHAND: nenhum arquivo de máscara em {mask_dir}.")
print(f"exemplo de máscara: {mask_files[0].name} (primeiras {min(5, len(mask_files))} serão amostradas)")
coverages = []
for mf in mask_files[:5]:
    mask = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"FreiHAND: não consegui ler a máscara {mf}.")
    if mask.ndim != 2:
        raise RuntimeError(f"FreiHAND: máscara {mf} não é 2D (shape {mask.shape}).")
    hand = mask > 127  # regra de uso: binarizar com limiar 127, nunca usar o JPEG cru
    cov = float(hand.mean())
    coverages.append(cov)
    rgb = mask_dir.parent / "rgb" / mf.name
    print(f"  {mf.name}: shape {mask.shape}, cobertura mão (limiar 127) {cov:.4f}, RGB pareado: {'OK' if rgb.exists() else 'AUSENTE'}")
if all(c == 0.0 for c in coverages):
    raise RuntimeError("FreiHAND: todas as máscaras amostradas têm cobertura 0 — nenhuma mão detectada. Não prossiga sem investigar.")
if all(c >= 0.99 for c in coverages):
    raise RuntimeError("FreiHAND: todas as máscaras amostradas são quase 100% mão — formato inesperado. Não prossiga sem investigar.")
print("FreiHAND OK para checagem: training/mask/ existe, máscara 2D, limiar 127, cobertura de mão plausível.")


## 4. Config e contagem de pixels

Lê `training/config.json` e conta pixels por classe no treino.
Compare com os pesos da config (CE [1, 2, 3, 10]): se a contagem real
divergir muito, recalibre os pesos e registre a mudança em
`training_config.json` — nunca ajustando peso olhando o split de teste.


In [ ]:
import sys, json
sys.path.insert(0, str(REPO))
from training.train import load_config, pair_images_masks, pixel_counts, compute_class_weights

cfg = load_config(REPO / "training" / "config.json")
print("modelo:", cfg["model"]["architecture"], cfg["model"]["backbone"])
print("classes:", cfg["model"]["classes"])
print("loss:", cfg["loss"]["formula"])
pairs = pair_images_masks(DATA / "train")
counts = pixel_counts(pairs)
print("pixels por classe (treino):", counts.tolist())
print("pesos calculados (referência):", [round(w, 2) for w in compute_class_weights(counts)])
print("pesos da config (usados):", cfg["loss"]["cross_entropy_class_weights"])


## 5. Treino

Chama `training/train.py`, que é a única fonte de verdade do treino
(modelo, loss, métricas, ONNX). O notebook é só um invólucro
reproduzível em cima dele. Em caso de OOM, o script sugere batch 4
com acumulação — não reduza a resolução: traço fino depende dela.

**Para testar, troque só `EPOCHS` abaixo:** `EPOCHS = None` usa o padrão
da config (30 épocas); `EPOCHS = 2` roda um teste curto. Não edite o
comando de treino.


In [ ]:
# Treino — EPOCHS é a única variável a trocar para testar.
EPOCHS = None  # None = usa a config (30 épocas); para teste, troque para 2

import subprocess, sys
cmd = [
    sys.executable,
    str(REPO / "training" / "train.py"),
    "--config", str(REPO / "training" / "config.json"),
    "--data-dir", str(DATA),
    "--out-dir", "/kaggle/working/out",
]
if EPOCHS is not None:
    cmd += ["--epochs", str(EPOCHS)]
print("comando:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 6. Métricas e cenários fixos

`metrics.json` traz IoU, Dice, precision e recall por classe, a
métrica de ink com tolerância (2-3 px) e a validação ONNX.
**Nunca reporte só accuracy global:** com fundo dominante, ela mente.
Os 15 cenários fixos (folha limpa, mão cobrindo canto/centro/traço,
sombra forte, traço fino de lápis, etc.) têm predições salvas para
comparação entre versões — o avaliador de cenários é o próximo passo
do agente de validação, fora deste notebook.


In [ ]:
import json
from pathlib import Path
metrics_path = Path("/kaggle/working/out/metrics.json")
if not metrics_path.exists():
    raise FileNotFoundError(
        "metrics.json não encontrado em /kaggle/working/out. O treino não completou "
        "ou salvou em outro lugar — verifique a célula de treino antes de ler métricas."
    )
m = json.load(open(metrics_path))
print("val_mean_iou:", round(m["mean_iou"], 4))
for cls in ("background", "paper", "hand", "ink"):
    d = m[cls]
    print(f"{cls:11s} IoU {d['iou']:.3f}  Dice {d['dice']:.3f}  P {d['precision']:.3f}  R {d['recall']:.3f}")
print("validação ONNX:", m.get("onnx_validation"))
print("artefatos:", sorted(p.name for p in Path("/kaggle/working/out").iterdir()))


## 7. Interface ONNX (para o Codex integrar em `draw/neural.py`)

- **Input:** `image`, tensor float32 `[1, 3, 512, 512]`, RGB,
  normalização ImageNet (mean 0.485/0.456/0.406, std 0.229/0.224/0.225)
  — idêntica ao contrato atual de `draw/neural.py`.
- **Output:** `logits`, `[1, 4, 512, 512]`, sem softmax embutido.
  Ordem das classes: 0=background, 1=paper, 2=hand, 3=ink.
- **Uso:** `paper_mask = (argmax == 1)`, `hand_mask = (argmax == 2)`,
  `ink_mask = (argmax == 3)`.
- **paper_presence:** DERIVADO da máscara (fração de pixels paper
  acima do limiar que a integração escolher), não uma cabeça auxiliar.
  O modelo antigo tinha 2 canais + `presence_logit`; o novo tem 4
  canais e cabeça única — a integração muda só a leitura da saída,
  não o pré-processamento.
- **Regra de rótulo:** mão sobre traço = hand. O modelo não enxerga
  através da mão; a memória temporal futura do Draw é que preservará
  o traço escondido, usando `ink_full_mask` (fora do treino) como
  referência de avaliação.
